# Ask a Question — Step-by-Step Pipeline Demo

Type a question in the next cell, then **Run All**. Each stage of the RAG
pipeline gets its own cell showing what it produced and how long it took; the
last cell prints the answer, the sources it came from, and a timing breakdown.

> **First run is slow.** The models load lazily on first use, so the *Embed*
> stage includes downloading/loading MiniLM and *Generate* includes
> flan-t5-large (~3 GB). Re-run the cells afterwards for warm timings.

In [10]:
# =============================================================
#  EDIT THIS CELL, THEN RUN ALL
# =============================================================

QUESTION = "What is data leakage and why is it hard to detect?"

DOCUMENT = "../data/Prototype/sample_lecture_notes.pdf"
# other things to try:
#   "../data/Prototype/sample_document.pdf"    (1-page PDF)
#   "../data/Prototype/sample_image.png"       (image  -> Qwen2-VL)
#   "../data/Prototype/sample_audio.m4a"       (audio  -> Whisper)

TOP_K = 3        # how many chunks to retrieve for the answer

## Setup

Imports the six pipeline stages and defines a small timer used by every stage below.

In [11]:
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

TIMINGS = {}


class step:
    """Times a stage and records it in TIMINGS."""

    def __init__(self, name):
        self.name = name

    def __enter__(self):
        print(f">> {self.name} ...")
        self.t0 = time.time()
        return self

    def __exit__(self, *exc):
        self.elapsed = time.time() - self.t0
        TIMINGS[self.name] = self.elapsed
        print(f"   done in {self.elapsed:.2f}s\n")


def source_of(chunk):
    """Human-readable provenance for a chunk: file name and page, when known."""
    src = getattr(chunk, "source_file", None) or "(unknown file)"
    page = getattr(chunk, "page", None)
    return f"{src}, page {page}" if page is not None else src


with step("0. Import"):
    from backend.pipeline.loader import load_file
    from backend.pipeline.preprocessor import preprocess
    from backend.pipeline.embedder import embed
    from backend.pipeline.vector_store import build_and_save, load
    from backend.pipeline.retriever import retrieve
    from backend.pipeline.generator import generate

print(f"Question : {QUESTION}")
print(f"Document : {DOCUMENT}")
print(f"Top-k    : {TOP_K}")

>> 0. Import ...
   done in 0.00s

Question : What is data leakage and why is it hard to detect?
Document : ../data/Prototype/sample_lecture_notes.pdf
Top-k    : 3


## Stage 1 — Load

`loader.py` pulls text out of the file. PDFs come back **per page** so that page
boundaries survive into chunking; images, audio and plain text come back as a
single string.

In [12]:
with step("1. Load"):
    loaded = load_file(DOCUMENT)

if isinstance(loaded, list):
    print(f"PDF - {len(loaded)} page(s) with extractable text\n")
    for p in loaded:
        preview = " ".join(p["text"].split())[:64]
        print(f"  page {p['page']:>2}  {len(p['text']):>5} chars   {preview}...")
else:
    print(f"Single text block - {len(loaded)} chars\n")
    print("  " + " ".join(loaded.split())[:400] + " ...")

[loader] Auto-detected '.pdf' → input_type='pdf'
[loader] Loading PDF → ../data/Prototype/sample_lecture_notes.pdf
[loader] PDF loaded — 6 pages (6 with text), 9342 characters


>> 1. Load ...
   done in 0.01s

PDF - 6 page(s) with extractable text

  page  1   1686 chars   Lecture 3 - Supervised Learning Supervised learning is the branc...
  page  2   1626 chars   Lecture 4 - Overfitting, Regularisation and Validation A model o...
  page  3   1586 chars   Lecture 5 - Evaluation Metrics Accuracy, the fraction of predict...
  page  4   1535 chars   Lecture 6 - Feature Engineering and Data Leakage Feature enginee...
  page  5   1411 chars   Lecture 7 - The Bias-Variance Decomposition The expected error o...
  page  6   1498 chars   Lecture 8 - Gradient Descent Variants Batch gradient descent com...


## Stage 2 — Preprocess

`preprocessor.py` cleans the text and splits it into overlapping 220-token
windows (40-token overlap). For PDFs each page is windowed **independently**, so
no chunk spans a page boundary, and every chunk records the file and page it
came from.

In [13]:
with step("2. Preprocess"):
    chunks = preprocess(loaded)

print(f"{len(chunks)} chunks - 220-token windows, 40-token overlap\n")

SHOW = 12
for i, c in enumerate(chunks[:SHOW]):
    print(f"[{i:>2}]  {source_of(c)}   ({len(c)} chars)")
    print(f"      {c[:110]}...")
if len(chunks) > SHOW:
    print(f"\n  ... and {len(chunks) - SHOW} more")

pages_used = sorted({c.page for c in chunks if c.page is not None})
if pages_used:
    print(f"\nChunks per page: "
          f"{ {p: sum(1 for c in chunks if c.page == p) for p in pages_used} }")

>> 2. Preprocess ...
   done in 0.01s

12 chunks - 220-token windows, 40-token overlap

[ 0]  sample_lecture_notes.pdf, page 1   (1179 chars)
      lecture 3 - supervised learning supervised learning is the branch of machine learning in which a model is trai...
[ 1]  sample_lecture_notes.pdf, page 1   (754 chars)
      age. the distinction matters because it determines which loss function is appropriate. classification typicall...
[ 2]  sample_lecture_notes.pdf, page 2   (1128 chars)
      lecture 4 - overfitting, regularisation and validation a model overfits when it captures noise in the training...
[ 3]  sample_lecture_notes.pdf, page 2   (715 chars)
      neural networks, randomly disables a fraction of units during each training step, forcing the network not to r...
[ 4]  sample_lecture_notes.pdf, page 3   (1112 chars)
      lecture 5 - evaluation metrics accuracy, the fraction of predictions that are correct, is the most intuitive m...
[ 5]  sample_lecture_notes.pdf, page 3   (67

## Stage 3 — Embed

`embedder.py` encodes every chunk into a 384-dimensional vector with
all-MiniLM-L6-v2.

In [14]:
with step("3. Embed"):
    embeddings = embed(chunks)

print(f"{embeddings.shape[0]} vectors x {embeddings.shape[1]} dimensions  "
      f"({embeddings.dtype})")

>> 3. Embed ...
   done in 0.07s

12 vectors x 384 dimensions  (float32)


## Stage 4 — Store

`vector_store.py` builds a FAISS `IndexFlatL2` index and saves it alongside the
chunks. Chunks are stored as `{text, source_file, page}` records, so provenance
survives the save/load round trip.

In [15]:
with step("4. Store"):
    build_and_save(embeddings, chunks)
    index, stored_chunks = load()

print(f"FAISS IndexFlatL2 - {index.ntotal} vectors")
print("Written to notebooks/data/processed/{index.faiss, chunks.json}\n")
print(f"Metadata survived the round trip: chunk[0] is from {source_of(stored_chunks[0])}")

>> 4. Store ...
   done in 0.03s

FAISS IndexFlatL2 - 12 vectors
Written to notebooks/data/processed/{index.faiss, chunks.json}

Metadata survived the round trip: chunk[0] is from sample_lecture_notes.pdf, page 1


## Stage 5 — Retrieve

`retriever.py` embeds the question and pulls the `TOP_K` nearest chunks out of
the index — these, and only these, become the model's context.

In [16]:
with step("5. Retrieve"):
    retrieved = retrieve(QUESTION, index, stored_chunks, k=TOP_K)

print(f"Question: {QUESTION}\n")
for rank, c in enumerate(retrieved, start=1):
    print(f"#{rank}  -- {source_of(c)}")
    print(f"    {c[:320]}...")
    print()

>> 5. Retrieve ...
   done in 0.02s

Question: What is data leakage and why is it hard to detect?

#1  -- sample_lecture_notes.pdf, page 4
    lecture 6 - feature engineering and data leakage feature engineering is the process of turning raw observations into inputs a model can use. numerical features are often standardised so that each has zero mean and unit variance, which stops features measured on large scales from dominating distance - based methods and ...

#2  -- sample_lecture_notes.pdf, page 4
    label rather than a cause of it, such as using the number of treatments a patient received to predict whether they were diagnosed. leakage announces itself as results that are too good to be true : near - perfect validation scores that collapse the moment the model meets genuinely new data. the defence is procedural ra...

#3  -- sample_lecture_notes.pdf, page 2
    lecture 4 - overfitting, regularisation and validation a model overfits when it captures noise in the training data as 

## Stage 6 — Generate

`generator.py` fits the retrieved chunks into flan-t5-large's 1024-token input
budget and generates the answer. When the chunks don't all fit, the budget is
shared across them so every retrieved chunk stays represented rather than the
lowest-ranked ones being cut off.

In [17]:
with step("6. Generate"):
    answer = generate(QUESTION, retrieved)

print(answer)

>> 6. Generate ...
   done in 1.89s

it occurs whenever information that would not be available at prediction time leaks into the training features


## Answer, sources and timing

In [18]:
line = "=" * 74

print(line)
print("QUESTION")
print(line)
print(QUESTION)

print()
print(line)
print("ANSWER")
print(line)
print(answer)

print()
print(line)
print("SOURCES")
print(line)
seen = []
for c in retrieved:
    s = source_of(c)
    if s not in seen:
        seen.append(s)
for rank, s in enumerate(seen, start=1):
    print(f"  {rank}. {s}")

print()
print(line)
print("TIMING")
print(line)
total = sum(TIMINGS.values())
# The import step is a one-off kernel cost, not part of answering a question,
# so the per-stage shares are shown against the pipeline stages only.
pipeline = {k: v for k, v in TIMINGS.items() if not k.startswith("0.")}
pipeline_total = sum(pipeline.values())

for name, secs in TIMINGS.items():
    share = secs / pipeline_total if pipeline_total else 0
    if name.startswith("0."):
        print(f"  {name:<16}{secs:>8.2f}s     one-off kernel setup")
    else:
        print(f"  {name:<16}{secs:>8.2f}s  {share * 100:>5.1f}%  "
              f"{'#' * max(1, round(28 * share))}")
print(f"  {'':<16}{'':>8}   {'-' * 14}")
print(f"  {'ANSWER TIME':<16}{pipeline_total:>8.2f}s   (stages 1-6)")
print(f"  {'TOTAL':<16}{total:>8.2f}s   (including one-off setup)")

QUESTION
What is data leakage and why is it hard to detect?

ANSWER
it occurs whenever information that would not be available at prediction time leaks into the training features

SOURCES
  1. sample_lecture_notes.pdf, page 4
  2. sample_lecture_notes.pdf, page 2

TIMING
  0. Import           0.00s     one-off kernel setup
  1. Load             0.01s    0.4%  #
  2. Preprocess       0.01s    0.3%  #
  3. Embed            0.07s    3.4%  #
  4. Store            0.03s    1.3%  #
  5. Retrieve         0.02s    0.9%  #
  6. Generate         1.89s   93.8%  ##########################
                             --------------
  ANSWER TIME         2.02s   (stages 1-6)
  TOTAL               2.02s   (including one-off setup)


Timings above include lazy model loading on the first run of the kernel. Re-run
the stage cells (without restarting) to see steady-state numbers — typically the
*Generate* stage dominates, since flan-t5-large runs a full decode pass.